# 01: Scouting-Selected Player Archetypes

Prepare the player-season inputs, build the eleven fixed profiles, and export the player role mixtures. See the [results](../ARCHETYPE_RESULTS.md).

## 1. Setup


In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Allow execution from the repo root or the notebooks directory.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(REPO_ROOT))
import paths

EXPERIMENT = REPO_ROOT / "pooled_archetypes"
OUT = REPO_ROOT / "local_outputs" / "pooled_archetypes"
sys.path.insert(0, str(EXPERIMENT / "scripts"))
from fit_pooled_roles import fit_roles

players_full = paths.load(paths.PLAYERS_FULL, float_precision="round_trip")
print(f"Source player-seasons: {len(players_full):,}")

## 2. Sample rules and feature sets


In [ ]:
MIN_MINUTES = 200
FIRST_SEASON_START = 2021
ID_COLS = ["Player ID", "Season"]

# Reuse the sample rules from the earlier archetype comparison.
season_start = players_full["Season"].str[:4].astype(int)
eligible = (season_start >= FIRST_SEASON_START) & (players_full["Min"] >= MIN_MINUTES)
pf = players_full.loc[eligible].copy()
assert not pf.duplicated(ID_COLS).any(), "Resolve duplicate player-seasons before fitting"
assert pf[["Player ID", "Team ID", "Season"]].notna().all().all()

# Source-row keys align all local exports; they are never model features.
record_ids = pd.Index("players_full:" + (pf.index + 1).astype(str), name="Record ID")
print(f"Eligible player-seasons: {len(pf):,}")
print(pf["Season"].value_counts().sort_index())


In [ ]:
BASELINE_FEATURES = [
    'Height (in)',
    'Usage %',
    'TS %',
    'FT Rate',
    'OReb %',
    'DReb %',
    'Ast %',
    'TO %',
    'Stl %',
    'Blk %',
    'Ast/TO',
    'Transition Shot %',
    'Blocked Per FGA',
    'Fouls Drawn Per FGA',
    '2nd Chance Pts % Pts',
    'Fast Break Pts % Pts',
    'Paint Pts % Pts',
    '3PM % Pts',
    '% Of Team Ast',
    '% Of Team FGA',
    '% Of Team FTA',
    '% Of Team TO',
    'PF %',
    '% Ast Rim',
    '% Ast Paint',
    'Rim Assisted %',
    'FGA Freq % Halfcourt',
    '% Ast Dunk',
    'Dunk FGA Freq %',
    'Layup FGA Freq %',
    'Alley-Oop Ast/40',
    '% Ast3',
    '3PM Assisted %',
    'Pts Created/40',
    'Pts Scored Unassisted/40',
    '2PM Unassisted/40',
    '3PM Unassisted/40',
    'Shooting Foul Drawn %',
    'And1 %',
    'Ffl Drawn %',
    'Off Fouls Drawn %',
    'FT Ratio',
    'Blk Rcv %',
    'Individual Stop %',
    'Att1 Attempt Share %',
    'Stl + Blk Per PF',
    'Non-Rim 2 Assisted %',
    'Rim 2 Attempt Share %',
    '3P Attempt Share %',
]

# The reduced variant changes only these four inputs.
EXCLUDED_FROM_STYLE = [
    'TS %',
    'Pts Scored Unassisted/40',
    '2PM Unassisted/40',
    '3PM Unassisted/40',
]
STYLE_FEATURES = [c for c in BASELINE_FEATURES if c not in EXCLUDED_FROM_STYLE]
assert len(BASELINE_FEATURES) == 49 and len(STYLE_FEATURES) == 45


## 3. Check stored rates and repair lost precision

Five exported core rates have only a few distinct values because fractions were stored
at coarse precision. Multiplying those rounded values by 100 cannot restore variation.
Reconstruct them from the existing full-season counts and denominators, retaining the
original values for comparison. The formulas return percentage points below.

These are locally validated reconstructions, not exact recovery of the original API
values: minutes in the export are rounded. All other source fields keep their existing
precision and units; standardization later handles the mixed scales.


In [ ]:
REPAIRED_RATES = ["Usage %", "Ast %", "OReb %", "DReb %", "Blk %"]
original_rates = pf[REPAIRED_RATES].copy()
print("Distinct values before repair:")
print(original_rates.nunique())

def safe_ratio(numerator, denominator, multiplier=100):
    # Keep unavailable inputs and nonpositive denominators missing.
    return multiplier * numerator / denominator.where(denominator > 0)

prepared = pf.copy()
t = pf["Min Team"] / 5
m = pf["Min"]
prepared["Usage %"] = safe_ratio(
    (pf["FGA"] + 0.44 * pf["FTA"] + pf["TO"]) * t,
    m * (pf["FGA Team"] + 0.44 * pf["FTA Team"] + pf["TO Team"]), multiplier=1,
) * 100
prepared["Ast %"] = safe_ratio(pf["Ast"], pf["FGM Team"] * m / t - pf["FGM"], multiplier=1) * 100
prepared["OReb %"] = safe_ratio(pf["OReb"] * t, m * (pf["OReb Team"] + pf["DReb Opp"]), multiplier=1) * 100
prepared["DReb %"] = safe_ratio(pf["DReb"] * t, m * (pf["DReb Team"] + pf["OReb Opp"]), multiplier=1) * 100
prepared["Blk %"] = safe_ratio(pf["Blk"] * t, m * (pf["FGA Opp"] - pf["3PA Opp"]), multiplier=1) * 100
assert prepared[REPAIRED_RATES].notna().all().all()
assert prepared[REPAIRED_RATES].ge(0).all().all()
assert prepared[REPAIRED_RATES].le(100).all().all()
print("Distinct values after repair:")
print(prepared[REPAIRED_RATES].nunique())


## 4. Build the derived features

The three broad shot groups use matched classified attempts, not box-score FGA.
Rim and three-point shares enter clustering; non-rim twos are retained for interpretation.
The same reference-component approach is used for the Att1/non-Att1 split.

For assistance, combine makes and assisted makes before dividing. Undefined ratios
remain missing here. Small denominators are flagged later, not used to exclude players.


In [ ]:
def total(columns):
    return pf[columns].sum(axis=1, min_count=len(columns))

broad_fga = total(["Rim FGA", "Paint FGA", "Midrange FGA", "Corner 3 FGA", "Above Break 3 FGA"])
prepared["Rim 2 Attempt Share %"] = safe_ratio(pf["Rim FGA"], broad_fga)
prepared["Non-Rim 2 Attempt Share %"] = safe_ratio(total(["Paint FGA", "Midrange FGA"]), broad_fga)
prepared["3P Attempt Share %"] = safe_ratio(total(["Corner 3 FGA", "Above Break 3 FGA"]), broad_fga)

sequence_fga = total(["Att1 FGA", "Att2 FGA", "Att3 P FGA"])
prepared["Att1 Attempt Share %"] = safe_ratio(pf["Att1 FGA"], sequence_fga)
prepared["Non-Att1 Attempt Share %"] = safe_ratio(total(["Att2 FGA", "Att3 P FGA"]), sequence_fga)
prepared["Stl + Blk Per PF"] = safe_ratio(total(["Stl", "Blk"]), pf["PF"], multiplier=1)
prepared["Non-Rim 2 Assisted %"] = safe_ratio(
    total(["Paint FGM Assisted", "Midrange FGM Assisted"]), total(["Paint FGM", "Midrange FGM"])
)

assert np.allclose(prepared[["Rim 2 Attempt Share %", "Non-Rim 2 Attempt Share %", "3P Attempt Share %"]].sum(axis=1), 100)
assert np.allclose(prepared[["Att1 Attempt Share %", "Non-Att1 Attempt Share %"]].sum(axis=1), 100)


## 5. Retain interpretation fields and identifiers

Detailed zones and component statistics are retained for explaining the fitted profiles,
but do not enter clustering. Seven-zone fields combine matching left/right regions.
Their denominator excludes unclassified, behind-hoop and heave attempts.


In [ ]:
ZONES = {
    "Rim": ["Rim"],
    "Paint": ["Paint"],
    "Baseline 2": ["Left Baseline 2", "Right Baseline 2"],
    "Elbow 2": ["Left Elbow 2", "Right Elbow 2"],
    "Corner 3": ["Left Corner 3", "Right Corner 3"],
    "Wing 3": ["Left Wing 3", "Right Wing 3"],
    "Top of Key 3": ["Top of Key 3"],
}
zone_fga = total([f"{part} FGA" for parts in ZONES.values() for part in parts])
for zone, parts in ZONES.items():
    attempts = total([f"{part} FGA" for part in parts])
    makes = total([f"{part} FGM" for part in parts])
    prepared[f"Zone {zone} Shot %"] = safe_ratio(attempts, zone_fga)
    prepared[f"Zone {zone} FG %"] = safe_ratio(makes, attempts)


In [ ]:
INTERPRETATION_FIELDS = [
    '2P %',
    '3P %',
    'FT %',
    '3PA Rate %',
    'Assisted Makes %',
    'FTM % Pts',
    '2PM % Pts',
    '% Ast Midrange',
    'Paint Assisted %',
    'Midrange Assisted %',
    'Above Break 3 Assisted %',
    'Corner 3 Assisted %',
    'Dunk Assisted %',
    '% Ast2',
    '2PM Assisted %',
    'Pts Assisted %',
    'Ast2/40',
    'Ast3/40',
    'Zone Rim Shot %',
    'Zone Rim FG %',
    'Zone Paint Shot %',
    'Zone Paint FG %',
    'Zone Baseline 2 Shot %',
    'Zone Baseline 2 FG %',
    'Zone Elbow 2 Shot %',
    'Zone Elbow 2 FG %',
    'Zone Corner 3 Shot %',
    'Zone Corner 3 FG %',
    'Zone Wing 3 Shot %',
    'Zone Wing 3 FG %',
    'Zone Top of Key 3 Shot %',
    'Zone Top of Key 3 FG %',
    'Non-Att1 Attempt Share %',
    'Non-Rim 2 Attempt Share %',
    'TS %',
    'Pts Scored Unassisted/40',
    '2PM Unassisted/40',
    '3PM Unassisted/40',
]

CONTEXT_FIELDS = ["Player ID", "Season", "Player", "Pos", "Class", "Min/G", "Team ID", "Min"]
records = prepared[CONTEXT_FIELDS].set_axis(record_ids)
interpretation = prepared[INTERPRETATION_FIELDS].set_axis(record_ids)
baseline = prepared[BASELINE_FEATURES].set_axis(record_ids)
style = baseline[STYLE_FEATURES].copy()
assert baseline.index.equals(style.index)
assert baseline[STYLE_FEATURES].equals(style)


## 6. Missingness, coverage and low-information flags

Keep the agreed sample intact. A missing assisted-three percentage can indicate no made
threes, rather than zero assisted makes. Event counts below 20 are a diagnostic flag,
not a new eligibility rule or a validated reliability cutoff. PBP-minute coverage is
also recorded without changing the 200-minute restriction.


In [ ]:
missing = baseline.isna()
print("Missing clustering inputs:")
print(missing.sum()[missing.any()])
print(f"Complete observed rows: {(~missing.any(axis=1)).sum():,} / {len(baseline):,}")

support = pd.DataFrame(index=pf.index)
support["classified_assists"] = total(["Ast2", "Ast3"])
support["rim_makes"] = total(["Rim FGM Assisted", "Rim FGM Unassisted"])
support["three_point_makes"] = total(["3PM Assisted", "3PM Unassisted"])
support["nonrim_two_makes"] = total(["Paint FGM", "Midrange FGM"])
support["pbp_minutes_ratio"] = pf["Min (PBP)"] / pf["Min"]
support["low_pbp_minutes"] = support["pbp_minutes_ratio"] < 0.9
for column in ["classified_assists", "rim_makes", "three_point_makes", "nonrim_two_makes"]:
    support[f"{column}_below_20"] = support[column] < 20
support.index = record_ids
print("Diagnostic flags (not exclusions):")
print(support.select_dtypes("bool").sum())


## 7. Distribution and correlation checks

Review ranges, missing values and correlations without automatically pruning the locked
feature sets. Some ratios can legitimately exceed one. The initial analysis also checked
the distributions by season and reviewed long tails; no clipping was applied.


In [ ]:
print(baseline.describe().loc[["min", "50%", "max"]].round(3).T)
assert not np.isinf(baseline.to_numpy()).any()
assert baseline.nunique().gt(1).all()

correlation = baseline.corr()
upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool)).stack()
print("Strong pairwise correlations, retained for the initial comparison:")
print(upper[upper.abs() >= 0.9].sort_values())


## 8. Missing-value handling and scaling

For the initial exploratory comparison, fill each missing feature with its observed
pooled median, then use population-standard-deviation z-scores (`ddof=0`). Fit the
parameters once on the baseline and select the reduced columns from that same result.

Median replacements are computational placeholders, not recovered player behavior.
Keep missingness masks outside the model inputs; feature counts remain 49 and 45.
Preserve genuine zeros and valid extreme values. No nonlinear transformations, extra
family weights or feature exclusions are applied. For held-out evaluation, these
parameters would need to be fitted on the training sample only.


In [ ]:
medians = baseline.median()
assert medians.notna().all(), "A feature has no observed values"
imputed = baseline.fillna(medians)
means = imputed.mean()
scales = imputed.std(ddof=0)
assert scales.gt(0).all()

standardized = (imputed - means) / scales
style_standardized = standardized[STYLE_FEATURES].copy()
assert np.isfinite(standardized.to_numpy()).all()
assert np.allclose(standardized.mean(), 0, atol=1e-10)
assert np.allclose(standardized.std(ddof=0), 1, atol=1e-10)
assert np.array_equal(imputed.to_numpy()[~missing.to_numpy()], baseline.to_numpy()[~missing.to_numpy()])
print(f"Baseline: {standardized.shape}; reduced variant: {style_standardized.shape}")


## 9. Fit the eleven role mixtures

Average the selected player-seasons within each role. Give each stat group equal total weight and apply the center eligibility rules before fitting.

In [ ]:
player_records = records.assign(Team=pf["Team"].set_axis(record_ids))
height = style["Height (in)"]
three_attempts_per_100 = safe_ratio(pf["3PA"], pf["Poss"]).set_axis(record_ids)
weights, errors = fit_roles(
    style_standardized, player_records, height, style["3P Attempt Share %"], three_attempts_per_100
)
print(f"Fitted {len(weights):,} player-seasons across {weights.shape[1]} roles")

## 10. Map weights back to players and export the CSV

Save the descriptive player fields and role weights in the local results folder.

In [ ]:
export = player_records[["Player", "Season", "Team", "Pos", "Class"]].join(weights)
top_roles = np.argsort(-weights.to_numpy(), axis=1, kind="stable")
for rank in range(3):
    export[f"Top role {rank + 1}"] = weights.columns.to_numpy()[top_roles[:, rank]]

results = OUT / "results"
results.mkdir(parents=True, exist_ok=True)
export.to_csv(results / "player_archetype_results.csv", index=False, float_format="%.4f")
missing.to_csv(OUT / "missingness_mask.csv")
print(f"Saved {len(export):,} rows to {results.relative_to(REPO_ROOT)}")

## Notes

- The role names and player-season pools were selected before this fit.
- Height sets eligibility for the three center roles. The distance calculation uses 43 stats.
- The fit uses a 0.20 penalty on the weighted distances from a player to the individual role profiles.
- Close weights can swap rank, as with Bol Dengdit’s True Wing and True 5 shares.